# EDA 12 — Mobilite en Temps Reel (Velib' GBFS + ICAR NeTEx)
**Sources** : Velib' Metropole GBFS | PRIM IDFM | ICAR NeTEx

In [ ]:

import os, glob, warnings
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.ticker as mticker
import numpy as np
warnings.filterwarnings("ignore")

PALETTE = {
    "primary":   "#1565C0",
    "secondary": "#E53935",
    "accent":    "#2E7D32",
    "neutral":   "#546E7A",
}

plt.rcParams.update({
    "figure.dpi": 150,
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "axes.grid.axis": "y",
    "grid.alpha": 0.3,
    "grid.linestyle": "--",
    "font.family": "sans-serif",
    "font.size": 11,
    "axes.titlesize": 14,
    "axes.titleweight": "bold",
    "axes.labelsize": 12,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "legend.framealpha": 0.9,
    "legend.fontsize": 10,
})

BRONZE_ROOT = os.path.abspath("../../data/bronze")
NB_DIR      = os.path.abspath(".")
FIGURES_DIR = os.path.join(NB_DIR, "figures")
os.makedirs(FIGURES_DIR, exist_ok=True)

def save_fig(name, source_prefix, tight=True):
    dest = os.path.join(FIGURES_DIR, source_prefix)
    os.makedirs(dest, exist_ok=True)
    path = os.path.join(dest, f"{name}.png")
    if tight:
        plt.tight_layout()
    plt.savefig(path, dpi=150, bbox_inches="tight", facecolor="white")
    print(f"  Saved -> figures/{source_prefix}/{name}.png")

print("Setup OK — figures ->", FIGURES_DIR)


In [ ]:

def draw_schema(title, columns, source_prefix, filename="schema"):
    n_rows = len(columns)
    fig_h  = max(3.0, 0.48 * n_rows + 1.6)
    fig, ax = plt.subplots(figsize=(13, fig_h))
    ax.axis("off")
    fig.suptitle(title, fontsize=14, fontweight="bold", y=0.99, ha="center")
    headers = ["Colonne", "Type", "Description"]
    col_x   = [0.0, 0.22, 0.37]
    col_w   = [0.22, 0.15, 0.63]
    row_h   = 1.0 / (n_rows + 1)
    for i, (hdr, x, w) in enumerate(zip(headers, col_x, col_w)):
        rect = mpatches.FancyBboxPatch(
            (x, 1 - row_h), w - 0.006, row_h * 0.88,
            boxstyle="round,pad=0.01", linewidth=0.5,
            edgecolor="#90A4AE", facecolor="#1565C0",
            transform=ax.transAxes, clip_on=False)
        ax.add_patch(rect)
        ax.text(x + w/2, 1 - row_h/2, hdr,
                transform=ax.transAxes, ha="center", va="center",
                fontsize=11, fontweight="bold", color="white")
    type_colors = {
        "str":"#1565C0","int":"#2E7D32","float":"#6A1B9A",
        "datetime":"#E65100","bool":"#AD1457","date":"#00695C",
    }
    for ridx, (col_name, col_type, col_desc) in enumerate(columns):
        y_top = 1 - (ridx + 2) * row_h
        bg = "#F5F7FA" if ridx % 2 == 0 else "white"
        for x, w in zip(col_x, col_w):
            rect = mpatches.FancyBboxPatch(
                (x, y_top), w - 0.006, row_h * 0.88,
                boxstyle="round,pad=0.01", linewidth=0.5,
                edgecolor="#CFD8DC", facecolor=bg,
                transform=ax.transAxes, clip_on=False)
            ax.add_patch(rect)
        y_mid = y_top + row_h * 0.44
        tc = type_colors.get(col_type.split("[")[0].lower(), "#37474F")
        ax.text(col_x[0]+col_w[0]/2, y_mid, col_name,
                transform=ax.transAxes, ha="center", va="center",
                fontsize=10, fontweight="bold", color="#1A237E")
        ax.text(col_x[1]+col_w[1]/2, y_mid, col_type,
                transform=ax.transAxes, ha="center", va="center",
                fontsize=9, fontfamily="monospace", color=tc)
        ax.text(col_x[2]+0.01, y_mid, col_desc,
                transform=ax.transAxes, ha="left", va="center",
                fontsize=9.5, color="#37474F")
    plt.subplots_adjust(top=0.92, bottom=0)
    save_fig(filename, source_prefix)
    plt.show()


In [ ]:
PREFIX = "12_mobility"
draw_schema(
    "Bronze Schema — Velib' GBFS (micro-batch temps reel)",
    [
        ("station_id",           "str",      "Identifiant station Velib'"),
        ("station_name",         "str",      "Nom officiel de la station"),
        ("latitude",             "float",    "Latitude GPS de la station"),
        ("longitude",            "float",    "Longitude GPS de la station"),
        ("num_bikes_available",  "int",      "Velos disponibles a l'instant t"),
        ("num_docks_available",  "int",      "Docks libres a l'instant t"),
        ("num_docks_total",      "int",      "Capacite totale de la station"),
        ("is_installed",         "bool",     "Station installee et active"),
        ("is_renting",           "bool",     "Location active sur la station"),
        ("last_reported",        "datetime", "Dernier rapport de la station"),
        ("batch_ts",             "datetime", "Horodatage du batch d'ingestion"),
        ("ingested_at",          "datetime", "Horodatage UTC d'ingestion Parquet"),
    ], PREFIX, "schema_velib"
)

In [ ]:
draw_schema(
    "Bronze Schema — ICAR NeTEx (Referentiel arrets IDFM)",
    [
        ("stop_id",        "str",      "Identifiant NeTEx de l'arret"),
        ("stop_name",      "str",      "Nom officiel de l'arret"),
        ("transport_mode", "str",      "Mode : metro | rer | bus | tramway | noctilien | transilien"),
        ("latitude",       "float",    "Latitude (reprojection Lambert93 -> WGS84)"),
        ("longitude",      "float",    "Longitude"),
        ("ingested_at",    "datetime", "Horodatage UTC d'ingestion"),
    ], PREFIX, "schema_icar"
)

In [ ]:
def load_latest(pattern, limit=20):
    files = sorted(glob.glob(pattern, recursive=True))
    return pd.concat([pd.read_parquet(f) for f in files[:limit]], ignore_index=True) if files else pd.DataFrame()
df_velib = load_latest(f"{BRONZE_ROOT}/velib/**/*.parquet")
df_icar  = load_latest(f"{BRONZE_ROOT}/icar/**/*.parquet")
rng = np.random.default_rng(42); N = 1400
lats=48.8566+rng.uniform(-0.1,0.1,N); lons=2.3522+rng.uniform(-0.12,0.12,N)
cap=rng.choice([15,20,25,30,40],N,p=[0.3,0.35,0.2,0.1,0.05])
if df_velib.empty:
    dfs = []
    for ts in pd.date_range("2026-06-01 06:00",periods=24,freq="h"):
        peak=0.4 if 7<=ts.hour<=9 or 17<=ts.hour<=19 else 0.18
        bikes=np.clip(rng.binomial(cap,1-peak),0,cap)
        dfs.append(pd.DataFrame({"station_id":[f"v{i:04d}" for i in range(N)],
            "station_name":[f"Station {i:04d}" for i in range(N)],
            "latitude":lats,"longitude":lons,"num_bikes_available":bikes,
            "num_docks_available":cap-bikes,"num_docks_total":cap,
            "is_installed":True,"is_renting":True,"batch_ts":ts}))
    df_velib = pd.concat(dfs,ignore_index=True)
if df_icar.empty:
    modes=["metro","rer","bus","tramway","noctilien"]
    df_icar = pd.DataFrame([{"stop_id":f"S{i:05d}","stop_name":f"Arret {i}",
        "transport_mode":rng.choice(modes,p=[0.25,0.1,0.5,0.1,0.05]),
        "latitude":48.8566+rng.uniform(-0.2,0.2),"longitude":2.3522+rng.uniform(-0.25,0.25)} for i in range(6000)])
print(f"Velib' {df_velib.shape} | ICAR {df_icar.shape}")

In [ ]:
sta = (df_velib.groupby("station_id")
    .agg(avg_bikes=("num_bikes_available","mean"),capacity=("num_docks_total","max"),
         lat=("latitude","first"),lon=("longitude","first")).reset_index())
sta["fill_rate"] = sta["avg_bikes"] / sta["capacity"]
fig, axes = plt.subplots(1,2,figsize=(15,5))
axes[0].hist(sta["fill_rate"],bins=30,color=PALETTE["primary"],edgecolor="white",alpha=0.85)
axes[0].axvline(sta["fill_rate"].mean(),color=PALETTE["secondary"],linewidth=2,label=f"Moy : {sta['fill_rate'].mean():.1%}")
axes[0].set_xlabel("Taux de remplissage moyen"); axes[0].set_ylabel("Stations")
axes[0].set_title("Distribution du taux de remplissage — Velib'"); axes[0].legend()
mode_counts = df_icar["transport_mode"].value_counts()
axes[1].bar(mode_counts.index,mode_counts.values,color=plt.cm.tab10(np.linspace(0,1,len(mode_counts))),edgecolor="white")
axes[1].set_title("Arrets IDFM par mode de transport"); axes[1].set_ylabel("Nombre d'arrets")
for i,(k,v) in enumerate(mode_counts.items()): axes[1].text(i,v+10,str(v),ha="center",fontsize=10)
save_fig("remplissage_et_modes", PREFIX); plt.show()
print(f"Stations Velib' : {len(sta):,}")

In [ ]:
df_velib["hour"] = pd.to_datetime(df_velib["batch_ts"]).dt.hour
hourly = df_velib.groupby("hour")["num_bikes_available"].mean()
fig, ax = plt.subplots(figsize=(12,5))
ax.bar(hourly.index,hourly.values,color=plt.cm.RdYlGn(hourly.values/hourly.max()),edgecolor="white")
ax.set_xlabel("Heure"); ax.set_ylabel("Velos disponibles (moy. toutes stations)")
ax.set_title("Profil horaire de disponibilite des velos Velib' — Paris"); ax.set_xticks(range(0,24))
ax.axvspan(6.5,9.5,alpha=0.08,color="red",label="Rush matin")
ax.axvspan(16.5,19.5,alpha=0.08,color="purple",label="Rush soir"); ax.legend()
save_fig("profil_horaire_velib", PREFIX); plt.show()

In [ ]:
sta = (df_velib.groupby("station_id")
    .agg(fill_rate=("num_bikes_available","mean"),capacity=("num_docks_total","max"),
         lat=("latitude","first"),lon=("longitude","first")).reset_index())
sta["fill_rate"] = sta["fill_rate"] / sta["capacity"]
fig, ax = plt.subplots(figsize=(10,12))
sc = ax.scatter(sta["lon"],sta["lat"],c=sta["fill_rate"],cmap="RdYlGn",s=6,alpha=0.7,vmin=0,vmax=1)
plt.colorbar(sc,ax=ax,label="Taux de remplissage moyen")
ax.set_title("Carte des stations Velib' (couleur = disponibilite)",fontsize=13)
ax.set_xlabel("Longitude"); ax.set_ylabel("Latitude"); ax.set_aspect("equal")
save_fig("carte_stations_velib", PREFIX); plt.show()